# Lab 4 — Build the Audience, Then Buy It
Name: Artem Khailo | Group: KM-42

## Setup

In [12]:
import kagglehub
import pandas as pd
import matplotlib.pyplot as plt
DATA = kagglehub.dataset_download("shkarupylomaxim/adtech-lab4-tv-audiences/versions/2")
programs = pd.read_parquet(f"{DATA}/programs.parquet")
viewings = pd.read_parquet(f"{DATA}/viewings.parquet")
daily_weights = pd.read_parquet(f"{DATA}/daily_weights.parquet")
viewer_attributes = pd.read_parquet(f"{DATA}/viewer_attributes.parquet")
attributes = pd.read_parquet(f"{DATA}/attributes.parquet")
rate_card = pd.read_parquet(f"{DATA}/rate_card.parquet")
survey = pd.read_csv(f"{DATA}/purchase_survey.txt", sep="\t", dtype=str, keep_default_na=False)
survey_layout = pd.read_csv(f"{DATA}/purchase_survey_layout.txt", sep="\t", dtype=str)
acr_sessions = pd.read_parquet(f"{DATA}/acr_sessions.parquet")
hh_attributes = pd.read_parquet(f"{DATA}/hh_attributes.parquet")
reference_ratings = pd.read_parquet(f"{DATA}/reference_ratings.parquet")
universe = pd.read_parquet(f"{DATA}/universe.parquet")

Using Colab cache for faster access to the 'adtech-lab4-tv-audiences' dataset.


## Task 1.0 — Clean the survey file

In [13]:
answer_codes= survey_layout[survey_layout["RECTYPE"] == "B"]
code_list= list(answer_codes["KEYWORD"])

ages =viewer_attributes[viewer_attributes["attr_id"] == 300]
ages =ages.set_index("respondent_id")["value"]
members =viewer_attributes[viewer_attributes["attr_id"].isin([101, 102])]
adults =ages[ages.index.isin(members["respondent_id"])]
adults =adults[adults >= 18]
adult_ids =list(adults.index)

log= []
s= survey.copy()
is_junk= (s["RESPID"] == "RESPID") | s["RESPID"].str.startswith("TOTAL")
log.append(["Repeated header or footer line", is_junk.sum(), "dropped"])
s = s[~is_junk]
data_rows = len(s)
delivered = s.copy()

s["id_text"]= s["RESPID"].str.strip()
s["id_text"]= s["id_text"].str.replace(".0", "", regex=False)
s["id"]= s["id_text"].astype(int)
bad_id= s["id"].astype(str) != s["RESPID"]
log.append(["RESPID with spaces, a '.0' or a leading zero", bad_id.sum(), "fixed: read as a number"])

s["kw"]= s["KEYWORD"].str.strip()
s["kw"]= s["kw"].str.upper()
bad_kw= s["kw"] != s["KEYWORD"]
log.append(["KEYWORD in lower case or with a space", bad_kw.sum(), "fixed: trimmed, upper case"])

unknown_code= ~s["kw"].isin(code_list)
log.append(["KEYWORD not in the layout", unknown_code.sum(), "dropped"])
s = s[~unknown_code]

s["value"]= s["VALUE"].str.strip()
s["value"]= s["value"].str.lower()
yes_words= ["1", "1.0", "y", "yes", "true"]
is_yes= s["value"].isin(yes_words)
other_yes= is_yes & (s["value"] != "1")
log.append(["VALUE written another way (1.0, y, yes, TRUE)", other_yes.sum(), "fixed: read as yes"])
log.append(["VALUE not a yes (0, no, N or empty)", (~is_yes).sum(), "dropped"])
s= s[is_yes]

not_adult =~s["id"].isin(adult_ids)
log.append(["RESPID not an adult panel member", not_adult.sum(), "dropped"])
s =s[~not_adult]

repeated =s.duplicated(["id", "kw"])
log.append(["Same person and KEYWORD twice", repeated.sum(), "dropped the copy"])
s =s[~repeated]

pair_number ={"X710101": 1, "X710102": 2, "X710103": 3}
s["pairs"] =s["kw"].map(pair_number)
pair_counts_per_person = s.groupby("id")["pairs"].count()
two_pair_counts = (pair_counts_per_person > 1).sum()
max_pairs = s.groupby("id")["pairs"].max()
s["max_pairs"] =s["id"].map(max_pairs)
lower_count = s["pairs"]< s["max_pairs"]
log.append(["Two pair counts for one person", lower_count.sum(), "kept the higher one"])
s = s[~lower_count]

people = pd.DataFrame(index=adult_ids)
people["pairs"] =s.groupby("id")["pairs"].max()
people["pairs"] =people["pairs"].fillna(0)
people["pairs"]= people["pairs"].astype(int)
people["stride"] = people.index.isin(s[s["kw"] == "X710201"]["id"])
people["kestrel"] =people.index.isin(s[s["kw"] == "X710202"]["id"])
people["easywalk"] =people.index.isin(s[s["kw"] == "X710203"]["id"])
people["buyer"] =people.index.isin(s["id"])

print("Survey data rows:", data_rows)
print("Clean yes answers:", len(s))
print("Adult panel members:", len(adult_ids))
print("Non-buyers:", (~people["buyer"]).sum())
print("People with two pair counts:", two_pair_counts)

bad_records =pd.DataFrame(log, columns=["Bad record", "Rows", "What we did"])
print(bad_records.to_string(index=False))

your_count ={"X710100": people["buyer"].sum(),
              "X710101": (people["pairs"] ==1).sum(),
              "X710102": (people["pairs"] ==2).sum(),
              "X710103": (people["pairs"] ==3).sum(),
              "X710201": people["stride"].sum(),
              "X710202": people["kestrel"].sum(),
              "X710203": people["easywalk"].sum(),
              "X710298": s[s["kw"] == "X710298"]["id"].nunique(),
              "X710299": s[s["kw"] == "X710299"]["id"].nunique()}
check =answer_codes[["KEYWORD", "Text", "SAMPLE"]].copy()
check["Your count"] = check["KEYWORD"].map(your_count)
print(check.to_string(index=False))

raw_kestrel =(delivered["KEYWORD"] == "X710202").sum()
raw_buyer_share =delivered["RESPID"].nunique()/ len(adult_ids)
clean_buyer_share =people["buyer"].mean()
print("Kestrel buyers, as delivered / clean:", raw_kestrel, "/", people["kestrel"].sum())
print("Share of buyers, as delivered / clean:", round(raw_buyer_share, 3), "/", round(clean_buyer_share, 3))

Survey data rows: 18346
Clean yes answers: 15397
Adult panel members: 8010
Non-buyers: 3231
People with two pair counts: 113
                                   Bad record  Rows                What we did
               Repeated header or footer line     3                    dropped
 RESPID with spaces, a '.0' or a leading zero  1455    fixed: read as a number
        KEYWORD in lower case or with a space  1104 fixed: trimmed, upper case
                    KEYWORD not in the layout   738                    dropped
VALUE written another way (1.0, y, yes, TRUE)  1142         fixed: read as yes
          VALUE not a yes (0, no, N or empty)   742                    dropped
             RESPID not an adult panel member   423                    dropped
                Same person and KEYWORD twice   933           dropped the copy
               Two pair counts for one person   113        kept the higher one
KEYWORD            Text SAMPLE  Your count
X710100             Any   4779        4779

**Answer:** What was wrong: a repeated header and a footer line inside the data; RESPID with spaces, a '.0' or a leading zero; keywords in lower case or with a space; codes not in the layout; yes written as 1.0, y, yes or TRUE, and rows that were not a yes; ids that are not adult panel members; duplicate answers; 113 people with two pair counts. How far off the raw file would have been: 516 Kestrel buyers instead of 354, and a share of buyers of 0.885 instead of 0.597.

## Task 1.1 — Four definitions

In [14]:
def audience_size(ids):
    weights= daily_weights[daily_weights["respondent_id"].isin(ids)]
    daily_sums= weights.groupby("broadcast_date")["weight"].sum()
    return daily_sums.mean()

buyer_ids= list(people[people["buyer"]].index)
heavy_ids= list(people[people["pairs"] == 3].index)
kestrel_ids= list(people[people["kestrel"]].index)
adults_25_54= adults[(adults >= 25) & (adults <= 54)]
ids_25_54= list(adults_25_54.index)

print("Adults 18+ (M):", f"{audience_size(adult_ids)/ 1000000:.1f}")
print("Shoe buyers (M):", f"{audience_size(buyer_ids)/ 1000000:.1f}")
print("Heavy buyers, 3+ pairs (M):", f"{audience_size(heavy_ids)/ 1000000:.1f}")
print("Kestrel buyers (M):", f"{audience_size(kestrel_ids)/ 1000000:.1f}")
print("Adults 25-54 (M):", f"{audience_size(ids_25_54)/ 1000000:.1f}")
print("Kestrel buyers in panel (n):", len(kestrel_ids))

Adults 18+ (M): 252.6
Shoe buyers (M): 148.1
Heavy buyers, 3+ pairs (M): 49.6
Kestrel buyers (M): 10.0
Adults 25-54 (M): 123.7
Kestrel buyers in panel (n): 354


## Task 1.2 — Who are they?

In [15]:
men= viewer_attributes[viewer_attributes["attr_id"] == 201]["respondent_id"]
women= viewer_attributes[viewer_attributes["attr_id"]== 202]["respondent_id"]
age_18_34= adults[adults <= 34].index
age_35_54= adults[(adults >= 35) & (adults <= 54)].index
age_55_plus= adults[adults >= 55].index

groups = {"Men 18-34": age_18_34[age_18_34.isin(men)],
          "Men 35-54": age_35_54[age_35_54.isin(men)],
          "Men 55+": age_55_plus[age_55_plus.isin(men)],
          "Women 18-34": age_18_34[age_18_34.isin(women)],
          "Women 35-54": age_35_54[age_35_54.isin(women)],
          "Women 55+": age_55_plus[age_55_plus.isin(women)]}

audiences = {"Adults 18+": adult_ids,
             "Shoe buyers": buyer_ids,
             "Heavy buyers": heavy_ids,
             "Kestrel buyers": kestrel_ids,
             "Adults 25-54": ids_25_54}

adult_share= {}
for group_name, group_ids in groups.items():
    adult_share[group_name] = audience_size(group_ids) / audience_size(adult_ids)

rows= []
for audience_name, ids in audiences.items():
    total = audience_size(ids)
    ids = pd.Series(ids)
    for group_name, group_ids in groups.items():
        in_group = ids[ids.isin(group_ids)]
        share = audience_size(in_group) / total
        index = share / adult_share[group_name] * 100
        rows.append([audience_name, group_name, share, index])

profile= pd.DataFrame(rows, columns=["Audience", "Group", "Share", "Index"])
profile= profile.fillna(0)
profile["Share"]= profile["Share"].round(3)
profile["Index"]= profile["Index"].round(0).astype(int)
print(profile.to_string(index=False))

      Audience       Group  Share  Index
    Adults 18+   Men 18-34  0.144    100
    Adults 18+   Men 35-54  0.160    100
    Adults 18+     Men 55+  0.185    100
    Adults 18+ Women 18-34  0.144    100
    Adults 18+ Women 35-54  0.160    100
    Adults 18+   Women 55+  0.208    100
   Shoe buyers   Men 18-34  0.140     97
   Shoe buyers   Men 35-54  0.158     99
   Shoe buyers     Men 55+  0.186    101
   Shoe buyers Women 18-34  0.140     98
   Shoe buyers Women 35-54  0.178    111
   Shoe buyers   Women 55+  0.197     95
  Heavy buyers   Men 18-34  0.111     77
  Heavy buyers   Men 35-54  0.138     87
  Heavy buyers     Men 55+  0.189    102
  Heavy buyers Women 18-34  0.139     97
  Heavy buyers Women 35-54  0.226    142
  Heavy buyers   Women 55+  0.196     94
Kestrel buyers   Men 18-34  0.118     82
Kestrel buyers   Men 35-54  0.189    119
Kestrel buyers     Men 55+  0.126     68
Kestrel buyers Women 18-34  0.181    126
Kestrel buyers Women 35-54  0.265    166
Kestrel buyers  

**Answer:** For Kestrel I would hand the planner heavy buyers (3+ pairs). It is the client's own target, it is large nd its profile differs from adults (women 35-54 index 142, men 18-34 index 77). Kestrel buyers are only 354 people in the panel, too few for stable program ratings. Adults 25-54 would drop people 55+, who are 0.189 + 0.196 of heavy buyers. Shoe buyers look almost like all adults because most adults are buyers (148.1M of 252.6M), so their indexes stay between 95 and 111. A fused purchase attribute cannot tell you whether a program's viewers buy more shoes than people of the same age, sex and income who do not watch it: buying is linked to viewing only through demographics.

## Task 2.1 — Programs by audience

In [16]:
eight =[["NBC", "NFL Football", "Sunday Night Football (NBC)"],
         ["ABC", "NFL Football", "Monday Night Football (ABC)"],
         ["ABC", "Dancing with the Stars", "Dancing with the Stars (ABC)"],
         ["CBS", "NCIS", "NCIS (CBS)"],
         ["FNC", "Hannity", "Hannity (FNC)"],
         ["HGTV", "Unsellable Houses", "Unsellable Houses (HGTV)"],
         ["FOOD", "Halloween Wars", "Halloween Wars (FOOD)"],
         ["FX", "Grotesquerie", "Grotesquerie (FX)"]]
program_order =[]
parts =[]
for network, name, label in eight:
    one =programs[(programs["network_code"] == network) & (programs["program_name"] == name)]
    one = one[one["daypart"] =="Prime"].copy()
    one["program"] = label
    parts.append(one)
    program_order.append(label)
telecasts = pd.concat(parts)

def to_minutes(t):
    h, m =t.split(":")
    return int(h) *60 +int(m) -360

day_start =pd.to_datetime(telecasts["broadcast_date"]) + pd.Timedelta(hours=6)
telecasts["minute"] =(telecasts["start_ts"] - day_start).dt.total_seconds() / 60
telecasts["weekday"] =pd.to_datetime(telecasts["broadcast_date"]).dt.weekday

time_rows =rate_card[rate_card["program_name"].isna()]
time_rows =time_rows.explode("weekdays")
time_rows["weekday"] =time_rows["weekdays"].astype(int)
time_rows["start"] =time_rows["start_time"].apply(to_minutes)
time_rows["end"] =time_rows["end_time"].apply(to_minutes)
time_rows =time_rows[["network_code", "weekday", "start", "end", "cost_30s_usd"]]

priced =telecasts.merge(time_rows, on=["network_code", "weekday"])
priced =priced[(priced["minute"] >= priced["start"]) & (priced["minute"] < priced["end"])]
time_cost =priced.set_index("telecast_id")["cost_30s_usd"]

program_rows =rate_card[rate_card["program_name"].notna()]
program_rows =program_rows[["network_code", "program_name", "cost_30s_usd"]]
with_program =telecasts.merge(program_rows, on=["network_code", "program_name"])
program_cost =with_program.set_index("telecast_id")["cost_30s_usd"]

telecasts["cost"] =telecasts["telecast_id"].map(program_cost)
telecasts["cost"] =telecasts["cost"].fillna(telecasts["telecast_id"].map(time_cost))

people_views =viewings[viewings["resp_type"].isin(["R", "L"])]
parts = []
for i, t in telecasts.iterrows():
    v= people_views[people_views["network_code"] == t["network_code"]]
    v= v[(v["start_ts"] < t["end_ts"]) & (v["end_ts"] > t["start_ts"])].copy()
    start= v["start_ts"].clip(lower=t["start_ts"])
    end= v["end_ts"].clip(upper=t["end_ts"])
    v["minutes"]= (end - start).dt.total_seconds() / 60
    v["telecast_id"]= t["telecast_id"]
    parts.append(v[["telecast_id", "respondent_id", "weight", "minutes"]])
telecast_views= pd.concat(parts)

def rate(telecasts, ids):
    v= telecast_views[telecast_views["respondent_id"].isin(ids)].copy()
    v["weighted_minutes"]= v["weight"] * v["minutes"]
    weighted_minutes= v.groupby("telecast_id")["weighted_minutes"].sum()
    weights= daily_weights[daily_weights["respondent_id"].isin(ids)]
    day_size= weights.groupby("broadcast_date")["weight"].sum()
    result= telecasts[["telecast_id", "program"]].copy()
    result["viewers"]= telecasts["telecast_id"].map(weighted_minutes)
    result["viewers"]= result["viewers"].fillna(0)
    result["viewers"]= result["viewers"]/ telecasts["duration_min"]
    result["rating"]= result["viewers"]/ telecasts["broadcast_date"].map(day_size) * 100
    return result

adults_by_telecast= rate(telecasts, adult_ids)
target_by_telecast= rate(telecasts, heavy_ids)
adults_by_program= adults_by_telecast.groupby("program")[["viewers", "rating"]].mean()
target_by_program= target_by_telecast.groupby("program")[["viewers", "rating"]].mean()
cost_by_program= telecasts.groupby("program")["cost"].mean()

table= pd.DataFrame(index=program_order)
table["Adults rating"] =adults_by_program["rating"]
table["Target rating"] = target_by_program["rating"]
table["Target viewers (000)"] = target_by_program["viewers"] /1000
table["Composition"] = target_by_program["viewers"] / adults_by_program["viewers"]
table["Index"] = table["Target rating"] / table["Adults rating"] * 100
table["Cost per :30"] = cost_by_program
table["Adults CPM"] = table["Cost per :30"] / adults_by_program["viewers"] * 1000
table["Target CPM"] = table["Cost per :30"] / target_by_program["viewers"] * 1000

shown = table.copy()
shown["Adults rating"] =shown["Adults rating"].round(2)
shown["Target rating"] = shown["Target rating"].round(2)
shown["Target viewers (000)"] = shown["Target viewers (000)"].round(0).astype(int)
shown["Composition"] =shown["Composition"].round(3)
shown["Index"] = shown["Index"].round(0).astype(int)
shown["Cost per :30"] = shown["Cost per :30"].round(0).astype(int)
shown["Adults CPM"] = shown["Adults CPM"].round(0).astype(int)
shown["Target CPM"] = shown["Target CPM"].round(0).astype(int)
print(shown.to_string())
print("Target universe (M):", f"{audience_size(heavy_ids) / 1000000:.1f}")

                              Adults rating  Target rating  Target viewers (000)  Composition  Index  Cost per :30  Adults CPM  Target CPM
Sunday Night Football (NBC)            5.82           5.79                  2865        0.195     99        487200          33         170
Monday Night Football (ABC)            8.93           8.36                  4149        0.184     94        723600          32         174
Dancing with the Stars (ABC)           3.25           3.14                  1564        0.191     97        158600          19         101
NCIS (CBS)                             2.21           2.43                  1206        0.216    110        178800          32         148
Hannity (FNC)                          1.09           0.84                   417        0.152     77         39900          15          96
Unsellable Houses (HGTV)               0.50           0.64                   317        0.249    127         12900          10          41
Halloween Wars (FOOD)      

## Task 2.2 — Index, volume or price?

In [17]:
print("Highest index (program):", table["Index"].idxmax())
print("Most target viewers (program):", table["Target viewers (000)"].idxmax())
print("Lowest target CPM (program):", table["Target CPM"].idxmin())

Highest index (program): Unsellable Houses (HGTV)
Most target viewers (program): Monday Night Football (ABC)
Lowest target CPM (program): Unsellable Houses (HGTV)


**Answer:** Not the same program. Unsellable Houses (HGTV) has the highest index (127) and the lowest target CPM (\$41), but it delivers only 317 thousand target viewers. Monday Night Football (ABC) delivers the most, 4,149 thousand, with an index of 94. To a client who wants only the highest index I would say: the index shows how concentrated the target is in a program's audience, not how many of the target it reaches. Buying only Unsellable Houses would reach few of the 49.6M heavy buyers. Like the shampoo example, a high index on a small audience does not deliver the volume, so we choose programs by target CPM and target viewers and use the index only to compare similar options.

## Task 2.3 — Kestrel buyers

In [18]:
kestrel_by_telecast= rate(telecasts, kestrel_ids)
kestrel_by_program= kestrel_by_telecast.groupby("program")["rating"].mean()

kestrel_table= pd.DataFrame(index=program_order)
kestrel_table["Heavy-buyer index"]= table["Index"]
kestrel_table["Kestrel index"]= kestrel_by_program / table["Adults rating"] * 100
kestrel_table["Heavy-buyer index"]= kestrel_table["Heavy-buyer index"].round(0).astype(int)
kestrel_table["Kestrel index"]= kestrel_table["Kestrel index"].round(0).astype(int)
print(kestrel_table.to_string())

                              Heavy-buyer index  Kestrel index
Sunday Night Football (NBC)                  99             99
Monday Night Football (ABC)                  94             86
Dancing with the Stars (ABC)                 97             83
NCIS (CBS)                                  110             57
Hannity (FNC)                                77             43
Unsellable Houses (HGTV)                    127             77
Halloween Wars (FOOD)                       116             95
Grotesquerie (FX)                            96            130


**Answer:** The Kestrel index jumps much more (from 43 to 130, against 77 to 127 for heavy buyers). Part of it is real: Kestrel buyers skew to women under 55 (index 166 for women 35-54, 58 for women 55+ in Task 1.2), so older-skewing programs like Hannity (43) and NCIS (57) index low. But most of the jumping is noise: only 354 Kestrel buyers are in the panel, against 1,661 heavy buyers (Task 1.0), so each program's Kestrel rating rests on a few viewers, and one or two people more or less change the index a lot. I would trust the heavy-buyer index, because it rests on almost five times as many panel people. To check, count how many Kestrel panel members actually watched each program, and compute the index separately for the first and the second half of October: a real index stays about the same, noise does not.

## Task 3.1 — Weights

In [19]:
acr= acr_sessions.copy()
acr["start"]= pd.to_datetime(acr["start_utc"], utc=True)
acr["start"]= acr["start"].dt.tz_convert("America/New_York")
acr["start"]= acr["start"].dt.tz_localize(None)
acr["end"]= pd.to_datetime(acr["end_utc"], utc=True)
acr["end"]= acr["end"].dt.tz_convert("America/New_York")
acr["end"]= acr["end"].dt.tz_localize(None)
acr["bday"]= (acr["start"] - pd.Timedelta(hours=6)).dt.normalize()

tvs= acr[["tv_id", "hashed_ip"]].drop_duplicates()
households_per_ip= hh_attributes.groupby("hashed_ip").size()
tvs["n_hh"]=tvs["hashed_ip"].map(households_per_ip)
tvs["n_hh"] =tvs["n_hh"].fillna(0)
matched =tvs[tvs["n_hh"] == 1]
matched =matched.merge(hh_attributes, on="hashed_ip")
matched["head_age"]="Under 65"
matched.loc[matched["head_age_band"]=="65+", "head_age"] = "65+"
matched = matched[["tv_id", "head_age", "income_band"]]

acr=acr[acr["end"] - acr["start"] >= pd.Timedelta(minutes=1)]

tv_days = acr[["tv_id", "bday"]].drop_duplicates()
tv_days = tv_days.merge(matched, on="tv_id")
cell_counts = tv_days.groupby(["bday", "head_age", "income_band"]).size()
cell_counts = cell_counts.reset_index(name="tvs_in_cell")
tv_days = tv_days.merge(cell_counts, on=["bday", "head_age", "income_band"])
tv_days = tv_days.merge(universe, on=["head_age", "income_band"])
tv_days["weight"] = tv_days["tv_households"] / tv_days["tvs_in_cell"]
weights_per_day = tv_days.groupby("bday")["weight"].sum()

print("TVs matched to one household:", len(matched))
print("Weights sum per day (M):", f"{weights_per_day.min() / 1000000:.1f}", "to", f"{weights_per_day.max() / 1000000:.1f}")

live =acr[acr["session_type"] == "live"]
live =live[live["tv_id"].isin(matched["tv_id"])]
reference =reference_ratings.merge(programs, on="telecast_id")
rows=[]
group_rows = []
for i, t in reference.iterrows():
    s = live[live["network_code"] == t["network_code"]]
    s = s[(s["start"] < t["end_ts"]) & (s["end"] > t["start_ts"])].copy()
    start = s["start"].clip(lower=t["start_ts"])
    end = s["end"].clip(upper=t["end_ts"])
    s["minutes"] = (end - start).dt.total_seconds() / 60
    minutes_per_tv = s.groupby("tv_id")["minutes"].sum()
    watched_ids = minutes_per_tv[minutes_per_tv >= 1].index
    day = tv_days[tv_days["bday"] == pd.Timestamp(t["broadcast_date"])].copy()
    watched = day[day["tv_id"].isin(watched_ids)]
    unweighted = len(watched) / len(day) * 100
    weighted = watched["weight"].sum() / day["weight"].sum() * 100
    name = t["program_name"] + " (" + t["network_code"] + ")"
    rows.append([name, unweighted, weighted, t["hh_rating"]])
    day["watched"] = day["tv_id"].isin(watched_ids)
    by_age = day.groupby("head_age")["watched"].mean() * 100
    by_income = day.groupby("income_band")["watched"].mean() * 100
    row = {"Telecast": name}
    for group in ["Under 65", "65+"]:
        row[group] = by_age[group]
    for band in ["Under $25k", "$25k-$50k", "$50k-$75k", "$75k-$100k", "$100k-$150k", "$150k+"]:
        row[band] = by_income[band]
    group_rows.append(row)

weighted_table =pd.DataFrame(rows, columns=["Telecast", "Unweighted", "Weighted", "Reference"])
print(weighted_table.round(2).to_string(index=False))

gap_unweighted = (weighted_table["Reference"] - weighted_table["Unweighted"]).mean()
gap_weighted = (weighted_table["Reference"] - weighted_table["Weighted"]).mean()
print("Mean gap unweighted (points):", f"{gap_unweighted:.2f}")
print("Mean gap weighted (points):", f"{gap_weighted:.2f}")
print("Share of gap closed:", f"{(gap_unweighted - gap_weighted) / gap_unweighted:.3f}")
by_group_table = pd.DataFrame(group_rows)
print(by_group_table.round(2).to_string(index=False))

TVs matched to one household: 19607
Weights sum per day (M): 125.8 to 125.8
                                    Telecast  Unweighted  Weighted  Reference
                        MLB Postseason (FOX)       17.03     17.76      25.86
                          NFL Football (NBC)       22.62     23.75      33.14
                                  NCIS (CBS)        5.50      5.80      10.56
                               Hannity (FNC)        1.04      1.35       2.30
                    Unsellable Houses (HGTV)        2.12      2.08       3.69
        Halloween Baking Championship (FOOD)        0.95      0.90       1.63
                           Grotesquerie (FX)        2.17      2.11       3.53
The Real Housewives of New York City (BRAVO)        0.71      0.64       1.23
Mean gap unweighted (points): 3.73
Mean gap weighted (points): 3.44
Share of gap closed: 0.075
                                    Telecast  Under 65   65+  Under $25k  $25k-$50k  $50k-$75k  $75k-$100k  $100k-$150k  $150k+

## Task 3.2 — What weighting cannot fix
**Answer:** Weighting raised four ratings: MLB Postseason (FOX) from 17.03 to 17.76, NFL Football (NBC) from 22.62 to 23.75, NCIS (CBS) from 5.50 to 5.80 and Hannity (FNC) from 1.04 to 1.35. The other four went slightly down, for example Grotesquerie (FX) from 2.17 to 2.11. The four that rose are the ones that homes with a head 65+ watch much more than younger homes: 25.32 against 15.04 for MLB, 34.93 against 19.65 for NFL, 8.59 against 4.76 for NCIS and 3.49 against 0.44 for Hannity. For the other four the two groups are close or 65+ homes watch less (Grotesquerie 1.97 against 2.22, Bravo 0.34 against 0.80). The footprint under-represents 65+ homes (Lab 3), so weighting gives them more weight and lifts the programs they watch. Most of the gap is still there: the mean gap only fell from 3.73 to 3.44 points (0.075 closed). Weighting cannot fix viewing the smart TV never records: the feed sees one TV per home and misses the other sets in the house (Lab 3). It also cannot fix homes that differ from the US in ways we do not weight on: a smart-TV home with broadband is not like a home without them in the same cell (Lecture 4). Weighting by income barely moves these ratings because the ratings differ much less between income bands than between age groups: for NFL from 20.89 to 24.03 across the six income bands against 19.65 and 34.93 by age, for Hannity from 0.77 to 1.48 against 0.44 and 3.49. Moving weight between income cells therefore changes the average only a little.

## Task 4.1 — October predicts October

In [20]:
networks= ["CBS", "NBC", "ABC", "FOX", "FNC"]
prime= programs[(programs["daypart"] == "Prime") & (programs["network_code"].isin(networks))].copy()
prime["date"]= pd.to_datetime(prime["broadcast_date"])
prime= prime[prime["date"] <= "2024-10-28"]
prime["program"] =prime["program_name"] + " (" + prime["network_code"] + ")"

parts=[]
for i, t in prime.iterrows():
    v=people_views[people_views["network_code"] == t["network_code"]]
    v=v[(v["start_ts"] < t["end_ts"]) & (v["end_ts"] > t["start_ts"])].copy()
    start=v["start_ts"].clip(lower=t["start_ts"])
    end=v["end_ts"].clip(upper=t["end_ts"])
    v["minutes"]=(end - start).dt.total_seconds() / 60
    v["telecast_id"]=t["telecast_id"]
    parts.append(v[["telecast_id", "respondent_id", "weight", "minutes"]])
telecast_views=pd.concat(parts)

members=viewer_attributes[viewer_attributes["attr_id"].isin([101, 102])]
p2_ids=list(members["respondent_id"].unique())
prime["p2_rating"]=rate(prime, p2_ids)["rating"]
prime["heavy_rating"]=rate(prime, heavy_ids)["rating"]

history=prime[prime["date"] <= "2024-10-21"]
week=prime[prime["date"] >= "2024-10-22"].copy()
week= week[week["program"].isin(history["program"])]
p2_forecast= history.groupby("program")["p2_rating"].mean()
heavy_forecast= history.groupby("program")["heavy_rating"].mean()
week["p2_forecast"]= week["program"].map(p2_forecast)
week["heavy_forecast"]= week["program"].map(heavy_forecast)

p2 =week[week["p2_rating"] > 0].copy()
p2["error"]= (p2["p2_rating"] - p2["p2_forecast"]).abs() / p2["p2_rating"]
heavy =week[week["heavy_rating"] > 0].copy()
heavy["error"]= (heavy["heavy_rating"] - heavy["heavy_forecast"]).abs() / heavy["heavy_rating"]
biggest =p2.loc[p2["error"].idxmax()]

print("Telecasts forecast (n):", len(week))
print("MAPE, P2+ (%):", f"{p2['error'].mean() * 100:.1f}")
print("MAPE, heavy buyers (%):", f"{heavy['error'].mean() * 100:.1f}")
print("Biggest miss, P2+ (program, date):", biggest["program"] + ",", biggest["date"].strftime("%Y-%m-%d"))

Telecasts forecast (n): 101
MAPE, P2+ (%): 35.8
MAPE, heavy buyers (%): 38.6
Biggest miss, P2+ (program, date): Krapopolis (FOX), 2024-10-27


## Task 4.2 — Why it missed

In [21]:
misses = p2.sort_values("error", ascending=False).head(5)
misses_table = pd.DataFrame()
misses_table["Program"]= misses["program"]
misses_table["Date"]= misses["date"].dt.strftime("%Y-%m-%d")
misses_table["Forecast"]= misses["p2_forecast"].round(2)
misses_table["Actual"]= misses["p2_rating"].round(2)
print(misses_table.to_string(index=False))

airings= prime[prime["program"].isin(misses["program"])].copy()
airings = airings.sort_values(["program", "start_ts"])
airings_table = pd.DataFrame()
airings_table["Program"]= airings["program"]
airings_table["Date"]= airings["date"].dt.strftime("%Y-%m-%d")
airings_table["Day"] =airings["date"].dt.day_name()
airings_table["Start"] =airings["start_ts"].dt.strftime("%H:%M")
airings_table["Minutes"] =airings["duration_min"]
airings_table["Repeat"] =airings["is_repeat"]
airings_table["P2+ rating"] =airings["p2_rating"].round(2)
print(airings_table.to_string(index=False))

                   Program       Date  Forecast  Actual
          Krapopolis (FOX) 2024-10-27      1.06    0.22
Universal Basic Guys (FOX) 2024-10-27      0.68    0.26
        Fire Country (CBS) 2024-10-27      1.70    0.67
             Accused (FOX) 2024-10-22      0.95    0.37
  What Would You Do? (ABC) 2024-10-23      0.44    0.18
                   Program       Date       Day Start  Minutes  Repeat  P2+ rating
             Accused (FOX) 2024-10-01   Tuesday 21:00       60    True        1.87
             Accused (FOX) 2024-10-08   Tuesday 21:00       60   False        0.61
             Accused (FOX) 2024-10-15   Tuesday 21:00       60   False        0.38
             Accused (FOX) 2024-10-22   Tuesday 21:00       60   False        0.37
             Accused (FOX) 2024-10-23 Wednesday 22:00       60    True        2.14
        Fire Country (CBS) 2024-10-18    Friday 21:00       60   False        1.81
        Fire Country (CBS) 2024-10-19  Saturday 21:00       60    True        1.58


**Answer:** In all five misses the actual rating was far below the forecast, and in most of them the forecast, not the week, was the problem: a plain mean over all history airings mixes airings that are not comparable. Krapopolis (FOX, forecast 1.06, actual 0.22) aired in its usual Sunday 21:30 slot, which had 0.26 and 0.30 in history, but the mean also includes two Wednesday 22:00 repeats with 1.76 and 1.93, about six times the Sunday originals. Universal Basic Guys (FOX, 0.68 vs 0.26) also aired in its usual Sunday 20:30 slot with the same 0.26 as on 2024-10-06, but the mean includes 1.09 from the 2024-10-20 airing at 20:00. Accused (FOX, 0.95 vs 0.37): the new episodes had 0.61 and 0.38, close to the actual, but the mean includes a 2024-10-01 repeat with 1.87. Fire Country (CBS, 1.70 vs 0.67): in history it aired as 60-minute episodes (1.81 and 1.58), while on 2024-10-27 the CBS schedule shifted and it ran as two short pieces of 14 and 16 minutes, the second at 22:44. What Would You Do? (ABC, 0.44 vs 0.18): the mean includes a Monday airing after Dancing with the Stars (0.49), and the Wednesday slot itself fell from 0.38 to 0.18. The heavy-buyer forecast is less accurate (MAPE 38.6% against 35.8%) because heavy buyers are a smaller group than persons 2+, so each rating rests on fewer panel people and moves more by chance. A better forecast would forecast each slot separately and treat repeats apart from new episodes (or drop outlier airings), and it would use the known schedule for the week, such as a game that runs long and shifts the programs after it.

## Task 5.1 — The plan

In [22]:
spots= {"Dancing with the Stars (ABC)": 4,
         "NCIS (CBS)": 1,
         "Unsellable Houses (HGTV)": 8,
         "Halloween Wars (FOOD)": 8}

plan =pd.DataFrame()
plan["Program"] =list(spots.keys())
plan["Spots"] =list(spots.values())
plan["Cost"] =plan["Program"].map(table["Cost per :30"]) * plan["Spots"]
plan["Target impressions (000)"] = plan["Program"].map(table["Target viewers (000)"]) * plan["Spots"]

shown_plan = plan.copy()
shown_plan["Cost"] = shown_plan["Cost"].round(0).astype(int)
shown_plan["Target impressions (000)"] = shown_plan["Target impressions (000)"].round(0).astype(int)
print(shown_plan.to_string(index=False))
print("Plan cost ($):", round(plan["Cost"].sum()))
print("Plan target impressions (M):", f"{plan['Target impressions (000)'].sum() / 1000:.1f}")

                     Program  Spots   Cost  Target impressions (000)
Dancing with the Stars (ABC)      4 634400                      6257
                  NCIS (CBS)      1 178800                      1206
    Unsellable Houses (HGTV)      8 103200                      2538
       Halloween Wars (FOOD)      8  77600                      1295
Plan cost ($): 994000
Plan target impressions (M): 11.3


## Task 5.2 — Recommendation
**Answer:** We buy 4 spots in Dancing with the Stars (ABC), 1 in NCIS (CBS), 8 in Unsellable Houses (HGTV) and 8 in Halloween Wars (FOOD) for \$994,000, which delivers 11.3M heavy-buyer impressions. The two cable programs are the cheapest per heavy buyer (target CPM \$41 and \$60) and have high indexes (127 and 116), but their audiences are small, so Dancing with the Stars brings the volume (1,564 thousand target viewers per spot at a target CPM of \$101) and NCIS adds a network audience with an index of 110. We skip football: it has the most viewers, but the highest target CPM (\$170 and \$174) and an index below 100. The delivery is not sure: our October forecast missed heavy-buyer ratings by 38.6% on average (MAPE), and the five biggest misses were all below the forecast, so we should buy with an audience guarantee, and the networks then owe make-goods if the plan under-delivers. Only 0.191 to 0.249 of the viewers of these programs are heavy buyers (composition), so three quarters to four fifths of the money pays for people who are not heavy buyers. After the week we would measure the heavy-buyer ratings of the telecasts where our spots actually ran, compare the delivered impressions with the planned 11.3M and claim make-goods for any shortfall.